In [1]:
"""
This code loads the ETTm1 time-series dataset, selects seven transformer-related
features, and converts them into NumPy arrays for model training and testing.

It defines a sliding-window setup where each sample contains:
- m observed time steps used as input,
- k future time steps used to determine the target label.

The dataset is sorted by date, split chronologically into training and testing
regions, and overlapping window start indices are generated using the chosen
stride. Assertions ensure that training windows remain entirely before the
train/test cutoff and test windows begin only after the cutoff.

Each window is assigned a binary label based on whether the
future value of the OT feature exceeds the threshold tau. The build_arrays
function then constructs separate arrays for observed inputs, future values,
and labels for both training and testing sets.

Finally, the code prints the number of generated windows and the resulting
array shapes.
"""
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
from sklearn.linear_model import RidgeClassifierCV
CSV_PATH  = "~/data/ETDataset/ETT-small/ETTm1.csv"
m  = 4     
k  = 48     
tau = 13     
train_frac = 0.8   
stride = 1      
COLS = ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']
df = (pd.read_csv(pathlib.Path(CSV_PATH).expanduser(),
                  parse_dates=['date'])
        .sort_values('date'))
X = df[COLS].to_numpy(dtype=np.float32)   
T = len(X)
window = m + k
cutoff_idx = int(np.floor(train_frac * T))
train_starts = np.arange(0, cutoff_idx - window + 1, stride)
test_starts  = np.arange(cutoff_idx, T         - window + 1, stride)    
assert np.all(train_starts + window <= cutoff_idx)
assert np.all(test_starts >= cutoff_idx)
print(f"Total windows          : {T - window + 1}")
print(f"Train windows (overlap): {len(train_starts)}")
print(f"Test  windows (overlap): {len(test_starts)}")
def early_warning_label(win, tau=tau):
    ot_future = win[m:, 6]
    return int(ot_future.max() > tau)
def build_arrays(starts):
    n = len(starts)
    x_obs = np.empty((n, m, 7), dtype=np.float32)
    x_fut = np.empty((n, k, 7), dtype=np.float32)
    y     = np.empty((n,),    dtype=np.int64)
    for idx, i in enumerate(starts):
        w = X[i : i + window]
        x_obs[idx] = w[:m]
        x_fut[idx] = w[m:]
        y[idx]     = early_warning_label(w)
    return x_obs, x_fut, y
x_obs_train, x_fut_train, y_train = build_arrays(train_starts)
x_obs_test,  x_fut_test,  y_test  = build_arrays(test_starts)
print("\nShapes:")
print(" x_obs_train:", x_obs_train.shape)
print(" x_fut_train:", x_fut_train.shape)
print(" y_train    :", y_train.shape)
print(" x_obs_test :", x_obs_test.shape)
print(" x_fut_test :", x_fut_test.shape)
print(" y_test     :", y_test.shape)

/opt/anaconda3/lib/python3.12/site-packages/aeon/base/__init__.py:24: FutureWarning: The aeon package will soon be releasing v1.0.0 with the removal of legacy modules and interfaces such as BaseTransformer and BaseForecaster. This will contain breaking changes. See aeon-toolkit.org for more information. Set aeon.AEON_DEPRECATION_WARNING or the AEON_DEPRECATION_WARNING environmental variable to 'False' to disable this warning.
  warnings.warn(


Total windows          : 69629
Train windows (overlap): 55693
Test  windows (overlap): 13885

Shapes:
 x_obs_train: (55693, 4, 7)
 x_fut_train: (55693, 48, 7)
 y_train    : (55693,)
 x_obs_test : (13885, 4, 7)
 x_fut_test : (13885, 48, 7)
 y_test     : (13885,)


In [2]:
"""
This code prints the shapes of the prepared training and testing arrays to
confirm that the observed inputs, future sequences, and label vectors have
been created with the expected dimensions.

It then checks the class distribution of the binary labels in both the training
and testing sets. For each unique class label, it counts how many examples are
present and calculates the percentage of the dataset belonging to that class.
"""
print("Shapes:")
print(" x_obs_train:", x_obs_train.shape)
print(" x_fut_train:", x_fut_train.shape)
print(" y_train    :", y_train.shape)
print(" x_obs_test :", x_obs_test.shape)
print(" x_fut_test :", x_fut_test.shape)
print(" y_test     :", y_test.shape)
unique_train, counts_train = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique_train, counts_train):
    pct = cnt / len(y_train) * 100
    print(f"Train class {cls}: {cnt} examples ({pct:.1f}%)")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique_test, counts_test):
    pct = cnt / len(y_test) * 100
    print(f" Test class {cls}: {cnt} examples ({pct:.1f}%)")

Shapes:
 x_obs_train: (55693, 4, 7)
 x_fut_train: (55693, 48, 7)
 y_train    : (55693,)
 x_obs_test : (13885, 4, 7)
 x_fut_test : (13885, 48, 7)
 y_test     : (13885,)
Train class 0: 21067 examples (37.8%)
Train class 1: 34626 examples (62.2%)
 Test class 0: 12012 examples (86.5%)
 Test class 1: 1873 examples (13.5%)


In [3]:
"""
This code defines a helper function to balance a binary classification dataset
by randomly downsampling the larger class to match the size of the smaller
class.

It first identifies the indices of class 0 and class 1 examples, finds the
minority class size, and randomly selects that same number of samples from
each class without replacement. The selected indices are then combined,
shuffled, and used to return balanced versions of the observed inputs, future
sequences, and labels.

The function is applied to the training data using a fixed random seed so that
the downsampling process is reproducible. Finally, Counter is used to print
the updated class counts in y_train, confirming that the training set is now
balanced.
"""
def balance_downsample(x_obs, x_fut, y, random_state=None):
    rng = np.random.default_rng(random_state)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    n_min = min(len(idx0), len(idx1))
    sel0 = rng.choice(idx0, size=n_min, replace=False)
    sel1 = rng.choice(idx1, size=n_min, replace=False)
    sel = np.concatenate([sel0, sel1])
    rng.shuffle(sel)
    return x_obs[sel], x_fut[sel], y[sel]
x_obs_train, x_fut_train, y_train = balance_downsample(
    x_obs_train, x_fut_train, y_train, random_state=42
)
from collections import Counter
print(Counter(y_train))

Counter({1: 21067, 0: 21067})


In [4]:
"""
This code assigns the prepared training and testing arrays to clearer variable
names that distinguish between observed input sequences, future forecast
sequences, and target labels.

It then selects the best available compute device for PyTorch, prioritising
Apple Silicon MPS if available, then CUDA for an NVIDIA GPU, and finally falling
back to the CPU if no supported accelerator is detected. 

Finally, the to_int helper function ensures that label arrays are represented
as integer class labels. If the labels are one-hot encoded, it converts them
using argmax; otherwise, it casts the existing labels directly to integers.
"""
X_observed_train =  x_obs_train
X_forecast_train =  x_fut_train
y_train =  y_train
X_observed_test =  x_obs_test
X_forecast_test =   x_fut_test 
y_test = y_test
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")
def to_int(y):
    y = np.asarray(y)
    return np.argmax(y, axis=1) if (y.ndim == 2 and y.shape[1] > 1) else y.astype(int)  

Using device: mps


In [5]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan})
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 83/83 [00:02<00:00, 35.08it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9814    0.8010    0.8821     12012
           1     0.4144    0.9028    0.5680      1873

    accuracy                         0.8148     13885
   macro avg     0.6979    0.8519    0.7251     13885
weighted avg     0.9049    0.8148    0.8397     13885

AUROC (seed 0): 0.9355
AUPRC (seed 0): 0.7585

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 83/83 [00:02<00:00, 35.42it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.9617    0.8783    0.9181     12012
           1     0.4985    0.7758    0.6069      1873

    accuracy                         0.8645     13885
   macro avg     0.7301    0.8270    0.7625     13885
weighted avg     0.8992    0.8645    0.8761     13885

AUROC (seed 1): 0.9281
AUPRC (seed 1): 0.7362

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.9530    0.8825    0.9164     12012
           1     0.4888    0.7208    0.5825      1873

    accuracy                         0.8606     13885
   macro avg     0.7209    0.8016    0.7494     13885
weighted avg     0.8904    0.8606    0.8713     13885

AUROC (seed 2): 0.9124
AUPRC (seed 2): 0.7094

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 83/83 [00:02<00:00, 35.05it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9818    0.7933    0.8775     12012
           1     0.4058    0.9055    0.5605      1873

    accuracy                         0.8084     13885
   macro avg     0.6938    0.8494    0.7190     13885
weighted avg     0.9041    0.8084    0.8348     13885

AUROC (seed 3): 0.9336
AUPRC (seed 3): 0.7566

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 83/83 [00:02<00:00, 35.37it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9568    0.9075    0.9315     12012
           1     0.5542    0.7373    0.6328      1873

    accuracy                         0.8846     13885
   macro avg     0.7555    0.8224    0.7821     13885
weighted avg     0.9025    0.8846    0.8912     13885

AUROC (seed 4): 0.9305
AUPRC (seed 4): 0.7485

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.9355  AUPRC=0.7585
Seed 1: AUROC=0.9281  AUPRC=0.7362
Seed 2: AUROC=0.9124  AUPRC=0.7094
Seed 3: AUROC=0.9336  AUPRC=0.7566
Seed 4: AUROC=0.9305  AUPRC=0.7485


In [7]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc})
    else:
        print("Warning: only one probability column; skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan})
print("\n" + "#"*80)
print("Summary over seeds (ModernTCNForecastThenClassifyOfficial)")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 83/83 [00:03<00:00, 24.89it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 83/83 [00:02<00:00, 30.93it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9727    0.8311    0.8963     12012
           1     0.4398    0.8505    0.5798      1873

    accuracy                         0.8337     13885
   macro avg     0.7063    0.8408    0.7381     13885
weighted avg     0.9008    0.8337    0.8536     13885

AUROC (seed 0): 0.9273
AUPRC (seed 0): 0.7260

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.9437    0.9599    0.9517     12012
           1     0.7109    0.6327    0.6695      1873

    accuracy                         0.9157     13885
   macro avg     0.8273    0.7963    0.8106     13885
weighted avg     0.9123    0.9157    0.9136     13885

AUROC (seed 3): 0.9357
AUPRC (seed 3): 0.7605

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 83/83 [00:03<00:00, 25.51it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 83/83 [00:02<00:00, 30.76it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.9574    0.8685    0.9108     12012
           1     0.4716    0.7523    0.5797      1873

    accuracy                         0.8529     13885
   macro avg     0.7145    0.8104    0.7453     13885
weighted avg     0.8919    0.8529    0.8662     13885

AUROC (seed 4): 0.9140
AUPRC (seed 4): 0.7285

################################################################################
Summary over seeds (ModernTCNForecastThenClassifyOfficial)
################################################################################
Seed 0: AUROC=0.9273  AUPRC=0.7260
Seed 1: AUROC=0.8895  AUPRC=0.6347
Seed 2: AUROC=0.9284  AUPRC=0.7529
Seed 3: AUROC=0.9357  AUPRC=0.7605
Seed 4: AUROC=0.9140  AUPRC=0.7285


In [7]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 83/83 [00:20<00:00,  4.13it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.9480    0.9405    0.9442     12012
           1     0.6367    0.6690    0.6524      1873

    accuracy                         0.9039     13885
   macro avg     0.7923    0.8047    0.7983     13885
weighted avg     0.9060    0.9039    0.9049     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.9305
AUPRC (MC mean over 100 samples, seed 0) : 0.7613

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 83/83 [00:19<00:00,  4.19it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.9293    0.9827    0.9552     12012
           1     0.8242    0.5206    0.6381      1873

    accuracy                         0.9203     13885
   macro avg     0.8767    0.7516    0.7967     13885
weighted avg     0.9151    0.9203    0.9125     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.9318
AUPRC (MC mean over 100 samples, seed 1) : 0.7632

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.9500    0.9284    0.9391     12012
           1     0.5993    0.6866    0.6400      1873

    accuracy                         0.8958     13885
   macro avg     0.7746    0.8075    0.7895     13885
weighted avg     0.9027    0.8958    0.8987     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.9286
AUPRC (MC mean over 100 samples, seed 2) : 0.7466

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 83/83 [00:19<00:00,  4.15it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.9211    0.9879    0.9534     12012
           1     0.8553    0.4576    0.5962      1873

    accuracy                         0.9164     13885
   macro avg     0.8882    0.7227    0.7748     13885
weighted avg     0.9123    0.9164    0.9052     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.9272
AUPRC (MC mean over 100 samples, seed 3) : 0.7483

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 83/83 [00:19<00:00,  4.18it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.9591    0.8772    0.9163     12012
           1     0.4912    0.7603    0.5968      1873

    accuracy                         0.8614     13885
   macro avg     0.7252    0.8187    0.7566     13885
weighted avg     0.8960    0.8614    0.8732     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.9304
AUPRC (MC mean over 100 samples, seed 4) : 0.7579

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.9305  AUPRC=0.7613
Seed 1: AUROC=0.9318  AUPRC=0.7632
Seed 2: AUROC=0.9286  AUPRC=0.7466
Seed 3: AUROC=0.9272  AUPRC=0.7483
Seed 4: AUROC=0.9304  AUPRC=0.7579


In [8]:
"""
This code trains and evaluates a ROCKET classifier across five different random seeds.

The prepare_labels function standardises the target labels into a one-dimensional
integer array. It supports labels that are already one-dimensional, labels stored
as a single-column two-dimensional array, and one-hot encoded labels. If the
label array has an unsupported shape, it raises an error.

For each seed, the code resets the random state for reproducibility and creates
a RocketMultivariateLengths transformer. The transformer generates a large set
of random convolutional features from the observed multivariate time-series
inputs using several kernel lengths.

The ROCKET transformer is fitted on the observed training data, then used to
convert both training and testing sequences into feature matrices. A
RidgeClassifierCV model is then trained on the transformed training features,
using cross-validation to choose the best regularisation strength from a range
of alpha values.

After training, the classifier predicts test labels and produces decision scores
for the test set. Performance metrics are then computed and printed. 
"""
def prepare_labels(y):
    y = np.asarray(y)
    if y.ndim == 1:
        return y.astype(int)
    if y.ndim == 2:
        if y.shape[1] == 1:
            return y.ravel().astype(int)
        return y.argmax(axis=1).astype(int)
    raise ValueError(f"Unsupported y shape {y.shape}")
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ROCKET + RidgeClassifierCV - seed {seed}")
    print("="*80)

    set_seed(seed)
    rocket = RocketMultivariateLengths(
        n_kernels=10000,        
        lengths=(2, 3, 4),
        random_state= seed,
    )
    rocket.fit(X_observed_train.astype(np.float32))
    X_train_feat = rocket.transform(X_observed_train.astype(np.float32))
    X_test_feat  = rocket.transform(X_observed_test.astype(np.float32))
    y_train_labels = prepare_labels(y_train)
    y_test_labels  = prepare_labels(y_test)
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(X_train_feat, y_train_labels)
    y_pred = clf.predict(X_test_feat)
    decision_scores = clf.decision_function(X_test_feat)  
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_test_labels, y_pred, digits=4))
    auroc = roc_auc_score(y_test_labels, decision_scores)
    auprc = average_precision_score(y_test_labels, decision_scores)
    print(f"AUROC (seed {seed}): {auroc:.4f}")
    print(f"AUPRC (seed {seed}): {auprc:.4f}")
    results.append({"seed": seed, "auroc": auroc, "auprc": auprc})
print("\n" + "#"*80)
print("Summary over seeds (ROCKET + RidgeClassifierCV)")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
    )


Training ROCKET + RidgeClassifierCV - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.9258    0.9391    0.9324     12012
           1     0.5700    0.5174    0.5424      1873

    accuracy                         0.8822     13885
   macro avg     0.7479    0.7282    0.7374     13885
weighted avg     0.8778    0.8822    0.8798     13885

AUROC (seed 0): 0.8831
AUPRC (seed 0): 0.6012

Training ROCKET + RidgeClassifierCV - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.9232    0.9475    0.9352     12012
           1     0.5947    0.4944    0.5399      1873

    accuracy                         0.8864     13885
   macro avg     0.7590    0.7209    0.7376     13885
weighted avg     0.8789    0.8864    0.8819     13885

AUROC (seed 1): 0.8835
AUPRC (seed 1): 0.5959

Training ROCKET + RidgeClassifierCV - seed 2

Classification report (seed 2):
              pre